# sPUNet vs HPUNet Comparative Evaluation

This notebook compares the performance of two probabilistic models for lung nodule segmentation on the LIDC dataset. Both models operate on the same data but have different architectures.

**Key Evaluation Strategy:**
- **Case 1:** All cases (actual lesions + no lesion) - measures overall clinical performance
- **Case 2:** Only actual lesions - measures pure detection capability when lesions exist

The idea is to see how good each model is at both predicting "nothing there" and finding real lesions.

## Configuration
Update these paths for your system:

In [66]:
import importlib
import hpunet.data.dataset
importlib.reload(hpunet.data.dataset)
from hpunet.data.dataset import LIDCCropsDataset

In [67]:
# Model and data paths - UPDATE THESE!!
SPUNET_CHECKPOINT = "/home/cdev/git/HPU-Net/runs/runs/iwi9140h-project/spu/run_20250829_082129_1197064/spu_last.pth"  # Update this path
SPUNET_CONFIG = "/home/cdev/git/HPU-Net/runs/runs/iwi9140h-project/spu/run_20250829_082129_1197064/train_spu_lidc.json"         # Update this path
HPUNET_CHECKPOINT = "/home/cdev/git/HPU-Net/runs/runs/iwi9140h-project/hpu/run_20250829_082159_1197065/hpu_last.pth"  # Update this path  
HPUNET_CONFIG = "/home/cdev/git/HPU-Net/runs/runs/iwi9140h-project/hpu/run_20250829_082159_1197065/train_hpu_lidc.json"         # Update this path

DATA_ROOT = "/home/cdev/git/HPU-Net/data/lidc_crops"                      # Update this path
CSV_NAME = "test.csv"                               

# Evaluation parameters
OUTPUT_DIR = "/home/cdev/git/HPU-Net/runs/runs/iwi9140h-project/comparative_evaluation_results"
NUM_EXAMPLES = 5          # Number of cases to visualize
NUM_SAMPLES = 8          # Prior samples per model per case
DEVICE = "cuda"           
EVAL_SCOPE = "all"        # "all" or "visualize_only"
GED_CLAMP_NONNEG = True   

# viz settings
TITLE_FONTSIZE = 14
FOOTER_FONTSIZE = 12
FIGURE_DPI = 150

print("Config loaded - dont forget to update the paths above!")

Config loaded - dont forget to update the paths above!


## Setup Python Path
This is equivalent to `export PYTHONPATH="$(pwd)/src"` in terminal

In [68]:
import sys
from pathlib import Path

# add src directory to python path
# equivalent to: export PYTHONPATH="$(pwd)/src" from HPU-Net root
# we need to go up to project root first
current_dir = Path.cwd()
print(f"current directory: {current_dir}")

# find HPU-Net root (go up until we find it)
project_root = current_dir
while project_root.name != "HPU-Net" and project_root.parent != project_root:
    project_root = project_root.parent

if project_root.name != "HPU-Net":
    # fallback: assume we're somewhere under HPU-Net, go up 3 levels
    project_root = current_dir.parent.parent.parent

src_path = project_root / "src"
print(f"project root: {project_root}")
print(f"adding to path: {src_path}")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    print("✅ added src to python path")
else:
    print("src already in python path")

# verify the path is correct by checking if hpunet exists
if (src_path / "hpunet").exists():
    print("✅ hpunet module found at correct path")
else:
    print("❌ hpunet module not found - check paths!")
    print(f"looking for: {src_path / 'hpunet'}")

current directory: /home/cdev/git/HPU-Net/src/hpunet/eval
project root: /home/cdev/git/HPU-Net
adding to path: /home/cdev/git/HPU-Net/src
src already in python path
✅ hpunet module found at correct path


## Imports and Setup
Loading all the stuff we need...

In [69]:
from __future__ import annotations
from typing import List, Optional, Dict, Tuple
import csv

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# model imports - should work now with pythonpath set
from hpunet.data.dataset import LIDCCropsDataset
from hpunet.models.spu_net import sPUNet
from hpunet.models.hpu_net import HPUNet
from hpunet.utils.config import load_config


# setup device and dirs
device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"using device: {device}")
print(f"output directory: {output_dir}")

using device: cuda
output directory: /home/cdev/git/HPU-Net/runs/runs/iwi9140h-project/comparative_evaluation_results


## Utility Functions
Helper functions for tensor conversion and metric computation

In [70]:
def tensor_to_numpy_mask(tensor: torch.Tensor, threshold: float = 0.5) -> np.ndarray:
    """convert logits/prob tensor to binary mask"""
    t = tensor.detach()
    if t.dim() == 4:
        t = t.squeeze(0).squeeze(0)
    elif t.dim() == 3:
        t = t.squeeze(0)
    arr = t.cpu().numpy().astype(np.float32)
    
    # sigmoid if needed
    if arr.max() > 1.0 or arr.min() < 0.0:
        arr = 1.0 / (1.0 + np.exp(-arr))
    
    return (arr > threshold).astype(np.uint8)


def tensor_to_numpy_ct_image(tensor: torch.Tensor) -> np.ndarray:
    """convert CT tensor to numpy for display"""
    t = tensor.detach()
    if t.dim() == 4:
        t = t.squeeze(0).squeeze(0)
    elif t.dim() == 3:
        t = t.squeeze(0)
    arr = t.cpu().numpy().astype(np.float32)
    
    # rescale to 0-1
    vmin, vmax = float(arr.min()), float(arr.max())
    if vmax > vmin:
        arr = (arr - vmin) / (vmax - vmin)
    else:
        arr = np.zeros_like(arr)
    
    return arr


def is_empty_mask(mask: np.ndarray) -> bool:
    """check if mask is totally black"""
    return mask.sum() == 0


def iou_binary(a: np.ndarray, b: np.ndarray) -> float:
    """basic IoU computation"""
    inter = np.logical_and(a > 0, b > 0).sum(dtype=np.float64)
    union = np.logical_or(a > 0, b > 0).sum(dtype=np.float64)
    
    if union == 0.0:
        return 1.0  # both empty = perfect
    
    return float(inter / union)


def pairwise_iou(set_A: List[np.ndarray], set_B: List[np.ndarray]) -> np.ndarray:
    """compute IoU matrix between two sets"""
    mat = np.zeros((len(set_A), len(set_B)), dtype=np.float64)
    for i, a in enumerate(set_A):
        for j, b in enumerate(set_B):
            mat[i, j] = iou_binary(a, b)
    return mat


print("utility functions ready")

utility functions ready


## GED² Computation
Generalized Energy Distance using IoU as the base metric

In [71]:
def ged2_iou(set_S: List[np.ndarray], set_Y: List[np.ndarray], clamp_nonneg: bool = True) -> Dict[str, float]:
    """
    GED² using IoU distance
    GED² = 2*E[d(S,Y)] - E[d(S,S')] - E[d(Y,Y')]
    where d = 1 - IoU
    """
    if len(set_S) == 0 or len(set_Y) == 0:
        return {
            "GED2": float("nan"), 
            "E_IoU_SY": float("nan"), 
            "E_IoU_YY": float("nan"), 
            "E_IoU_SS": float("nan")
        }

    # compute pairwise IoU matrices
    iou_SY = pairwise_iou(set_S, set_Y)
    iou_SS = pairwise_iou(set_S, set_S)
    iou_YY = pairwise_iou(set_Y, set_Y)
    
    # expectations (includes diagonal)
    E_IoU_SY = float(iou_SY.mean())
    E_IoU_SS = float(iou_SS.mean())
    E_IoU_YY = float(iou_YY.mean())
    
    # convert to distance and compute GED²
    E_d_SY = 1.0 - E_IoU_SY
    E_d_SS = 1.0 - E_IoU_SS
    E_d_YY = 1.0 - E_IoU_YY
    
    GED2 = 2.0 * E_d_SY - E_d_SS - E_d_YY
    
    # clamp to avoid tiny negatives from sampling bias
    if clamp_nonneg and np.isfinite(GED2) and GED2 < 0.0:
        GED2 = 0.0

    return {
        "GED2": float(GED2), 
        "E_IoU_SY": E_IoU_SY, 
        "E_IoU_YY": E_IoU_YY, 
        "E_IoU_SS": E_IoU_SS
    }

print("GED² function ready")

GED² function ready


## Model Loading Functions
Loading both model architectures with their specific parameters

In [72]:
def load_spunet(ckpt_path: Path, device: torch.device) -> sPUNet:
    """load sPUNet from checkpoint"""
    ckpt = torch.load(ckpt_path, map_location=device)
    # sPUNet uses these params based on the original script
    model = sPUNet(in_ch=1, base=32, z_dim=6).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    print(f"loaded sPUNet from step {ckpt.get('step', 'unknown')}")
    return model


def load_hpunet(ckpt_path: Path, device: torch.device) -> HPUNet:
    """load HPUNet from checkpoint"""
    ckpt = torch.load(ckpt_path, map_location=device)
    # HPUNet uses different params
    model = HPUNet(in_ch=1, base=24, z_ch=1, n_blocks=3).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    print(f"loaded HPUNet from step {ckpt.get('step', 'unknown')}")
    return model

print("model loading functions ready")

model loading functions ready


## Load Both Models
Actually loading the models here - make sure your paths are correct above!

In [73]:
print("loading models...")

spu_model = load_spunet(Path(SPUNET_CHECKPOINT), device)
hpu_model = load_hpunet(Path(HPUNET_CHECKPOINT), device)

# load configs too (might need them later)
spu_config = load_config(Path(SPUNET_CONFIG))
hpu_config = load_config(Path(HPUNET_CONFIG))

print("\nboth models loaded successfully!")

loading models...
loaded sPUNet from step 240000
loaded HPUNet from step 240000

both models loaded successfully!


## Dataset Setup
Using the same dataset for both models to ensure fair comparison

In [74]:
# setup dataset
csv_path = Path(DATA_ROOT) / CSV_NAME
dataset = LIDCCropsDataset(
    csv_path=csv_path,
    project_root=Path(DATA_ROOT).parent.parent,
    image_size=128,
    augment=False,  # no augmentation for eval
    seed=42,        # fixed seed for reproducibility
)

dataloader = DataLoader(
    dataset, 
    batch_size=1, 
    shuffle=False,  # keep order for comparison
    num_workers=2
)

print(f"loaded dataset with {len(dataset)} samples from {csv_path.name}")
print(f"will process {'all samples' if EVAL_SCOPE == 'all' else f'first {NUM_EXAMPLES} for visualization'}")

loaded dataset with 1980 samples from test.csv
will process all samples


## Inference Functions
These work with both model types since they have the same interface

In [75]:
@torch.no_grad()
def generate_reconstructions(model, image: torch.Tensor, grader_masks: torch.Tensor) -> List[torch.Tensor]:
    """generate posterior reconstructions for any model"""
    reconstructions = []
    for grader_idx in range(grader_masks.shape[1]):  # usually 4 graders
        grader_mask = grader_masks[:, grader_idx : grader_idx + 1, :, :].float()
        logits, _ = model(x=image, y_target=grader_mask, sample_posterior=True)
        reconstructions.append(logits)
    return reconstructions


@torch.no_grad()
def generate_samples(model, image: torch.Tensor, num_samples: int = 24) -> List[torch.Tensor]:
    """generate prior samples"""
    samples = []
    for _ in range(num_samples):
        logits, _ = model(x=image, y_target=None, sample_posterior=False)
        samples.append(logits)
    return samples


print("inference functions ready")

inference functions ready


## Metrics Computation Functions

The main idea here is to compute metrics for both evaluation cases:
- **Case 1**: Include everything (tests overall performance)
- **Case 2**: Only non-empty ground truth (tests lesion detection ability)

In [76]:
def compute_comparative_metrics_all_cases(
    spu_reconstructions: List[np.ndarray],
    hpu_reconstructions: List[np.ndarray], 
    grader_masks: List[np.ndarray],
    clamp_nonneg: bool = True
) -> Dict[str, float]:
    """
    Case 1: ALL cases included (actual lesions + no lesion)
    measures overall performance including "nothing there" prediction
    """
    
    if len(grader_masks) == 0:
        return {
            "num_graders": 0,
            "spu_GED2_all": float("nan"), "spu_E_IoU_SY_all": float("nan"),
            "hpu_GED2_all": float("nan"), "hpu_E_IoU_SY_all": float("nan"),
            "spu_E_IoU_SS_all": float("nan"), "spu_E_IoU_YY_all": float("nan"),
            "hpu_E_IoU_SS_all": float("nan"), "hpu_E_IoU_YY_all": float("nan")
        }
    
    # both models on all graders (including empty ones)
    spu_metrics = ged2_iou(spu_reconstructions, grader_masks, clamp_nonneg)
    hpu_metrics = ged2_iou(hpu_reconstructions, grader_masks, clamp_nonneg)
    
    return {
        "num_graders": len(grader_masks),
        "spu_GED2_all": spu_metrics["GED2"],
        "spu_E_IoU_SY_all": spu_metrics["E_IoU_SY"], 
        "spu_E_IoU_SS_all": spu_metrics["E_IoU_SS"],
        "spu_E_IoU_YY_all": spu_metrics["E_IoU_YY"],
        "hpu_GED2_all": hpu_metrics["GED2"],
        "hpu_E_IoU_SY_all": hpu_metrics["E_IoU_SY"],
        "hpu_E_IoU_SS_all": hpu_metrics["E_IoU_SS"], 
        "hpu_E_IoU_YY_all": hpu_metrics["E_IoU_YY"]
    }


def compute_comparative_metrics_nonempty_only(
    spu_reconstructions: List[np.ndarray],
    hpu_reconstructions: List[np.ndarray], 
    grader_masks: List[np.ndarray],
    clamp_nonneg: bool = True
) -> Dict[str, float]:
    """
    Case 2: only non-empty GT (actual lesions only)
    measures detection performance when lesions are present
    """
    
    # filter out empty GT masks
    non_empty_indices = [i for i, mask in enumerate(grader_masks) if not is_empty_mask(mask)]
    
    if len(non_empty_indices) == 0:
        return {
            "num_nonempty_graders": 0,
            "spu_GED2_nonempty": float("nan"), "spu_E_IoU_SY_nonempty": float("nan"),
            "hpu_GED2_nonempty": float("nan"), "hpu_E_IoU_SY_nonempty": float("nan"),
            "spu_E_IoU_SS_nonempty": float("nan"), "spu_E_IoU_YY_nonempty": float("nan"),
            "hpu_E_IoU_SS_nonempty": float("nan"), "hpu_E_IoU_YY_nonempty": float("nan")
        }
    
    # get only non-empty graders and corresponding reconstructions
    non_empty_graders = [grader_masks[i] for i in non_empty_indices]
    spu_recons_nonempty = [spu_reconstructions[i] for i in non_empty_indices]
    hpu_recons_nonempty = [hpu_reconstructions[i] for i in non_empty_indices]
    
    # compute metrics
    spu_metrics = ged2_iou(spu_recons_nonempty, non_empty_graders, clamp_nonneg)
    hpu_metrics = ged2_iou(hpu_recons_nonempty, non_empty_graders, clamp_nonneg)
    
    return {
        "num_nonempty_graders": len(non_empty_graders),
        "spu_GED2_nonempty": spu_metrics["GED2"],
        "spu_E_IoU_SY_nonempty": spu_metrics["E_IoU_SY"], 
        "spu_E_IoU_SS_nonempty": spu_metrics["E_IoU_SS"],
        "spu_E_IoU_YY_nonempty": spu_metrics["E_IoU_YY"],
        "hpu_GED2_nonempty": hpu_metrics["GED2"],
        "hpu_E_IoU_SY_nonempty": hpu_metrics["E_IoU_SY"],
        "hpu_E_IoU_SS_nonempty": hpu_metrics["E_IoU_SS"], 
        "hpu_E_IoU_YY_nonempty": hpu_metrics["E_IoU_YY"]
    }


def compute_individual_ious(
    reconstructions: List[np.ndarray], 
    grader_masks: List[np.ndarray]
) -> List[Optional[float]]:
    """
    compute IoU for each recon vs corresponding grader
    used for visualization - always computed
    """
    ious = []
    for i in range(len(reconstructions)):
        if i < len(grader_masks):
            grader = grader_masks[i]
            recon = reconstructions[i]
            ious.append(iou_binary(recon, grader))
        else:
            ious.append(None)
    
    return ious


print("metrics functions ready")

metrics functions ready


## Load Models

In [77]:
print("Loading models...")
spu_model = load_spunet(Path(SPUNET_CHECKPOINT), device)
hpu_model = load_hpunet(Path(HPUNET_CHECKPOINT), device)

# Load configs 
spu_config = load_config(Path(SPUNET_CONFIG))
hpu_config = load_config(Path(HPUNET_CONFIG))

print("Both models loaded successfully!")

Loading models...
loaded sPUNet from step 240000
loaded HPUNet from step 240000
Both models loaded successfully!


## Visualization Function

Creates side-by-side comparison showing both models on the same data.
The layout shows graders, then sPUNet results, then HPUNet results, then samples from both.

In [78]:
def create_comparative_visualization(
    ct_scan: np.ndarray,
    grader_masks: List[np.ndarray],
    spu_reconstructions: List[np.ndarray],
    hpu_reconstructions: List[np.ndarray],
    spu_ious: List[Optional[float]],
    hpu_ious: List[Optional[float]], 
    spu_samples: List[np.ndarray],
    hpu_samples: List[np.ndarray],
    save_path: Path
):
    """
    side-by-side comparison viz:
    Row 1: [CT] [grader1] [grader2] [grader3] [grader4] [empty]
    Row 2: [empty] [sPU_recon1] [sPU_recon2] [sPU_recon3] [sPU_recon4] [empty]  
    Row 3: [empty] [HPU_recon1] [HPU_recon2] [HPU_recon3] [HPU_recon4] [empty]
    Row 4: [sPU_sample1-6]
    Row 5: [HPU_sample1-6]
    """
    fig = plt.figure(figsize=(18, 15))
    
    def add_subplot(row, col, img, title, show_iou=False, iou_val=None):
        ax = fig.add_subplot(5, 6, row * 6 + col + 1)
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.set_title(title, fontsize=TITLE_FONTSIZE, loc="right")
        ax.axis("off")
        
        if show_iou:
            if iou_val is None:
                txt = "IoU=NA"
            elif np.isnan(iou_val):
                txt = "IoU=NA"
            else:
                txt = f"IoU={iou_val:.3f}"
            ax.text(1.0, -0.08, txt, transform=ax.transAxes, 
                   ha="right", va="top", fontsize=FOOTER_FONTSIZE)
        
        return ax
    
    # row 1: CT + graders
    add_subplot(0, 0, ct_scan, "CT")
    for i in range(min(4, len(grader_masks))):
        add_subplot(0, i + 1, grader_masks[i], f"grader {i+1}")
    
    # row 2: sPUNet reconstructions
    for i in range(min(4, len(spu_reconstructions))):
        title = f"sPU_recon {i+1}" if i > 0 else "sPUNet Reconstructions\nsPU_recon 1"
        add_subplot(1, i + 1, spu_reconstructions[i], title, show_iou=True, iou_val=spu_ious[i])
    
    # row 3: HPUNet reconstructions  
    for i in range(min(4, len(hpu_reconstructions))):
        title = f"HPU_recon {i+1}" if i > 0 else "HPUNet Reconstructions\nHPU_recon 1"
        add_subplot(2, i + 1, hpu_reconstructions[i], title, show_iou=True, iou_val=hpu_ious[i])
    
    # row 4: sPUNet samples
    for i in range(min(6, len(spu_samples))):
        title = f"sPU_s{i+1}" if i > 0 else "sPUNet Samples\nsPU_s1"
        add_subplot(3, i, spu_samples[i], title)
    
    # row 5: HPUNet samples
    for i in range(min(6, len(hpu_samples))):
        title = f"HPU_s{i+1}" if i > 0 else "HPUNet Samples\nHPU_s1"
        add_subplot(4, i, hpu_samples[i], title)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close()
    print(f"saved: {save_path}")


print("visualization function ready")

visualization function ready


## Main Evaluation Loop

This is where the magic happens - we run both models on the same data and collect all the metrics.

For each sample, we:
1. Generate reconstructions from both models  
2. Generate prior samples from both models
3. Compute metrics for both evaluation cases
4. Create comparative visualizations
5. Store everything for analysis

In [ ]:
print("starting comparative evaluation...\n")

# storage for results
all_case_metrics = []      # all cases in CSV scope
four_grader_metrics = []   # only cases with all 4 graders

num_visualized = 0

for row_idx, batch in enumerate(dataloader):
    if EVAL_SCOPE == "visualize_only" and num_visualized >= NUM_EXAMPLES:
        break
    
    if row_idx % 100 == 0:  # progress update
        print(f"processing row {row_idx}...")
    
    # get inputs
    image = batch["image"].to(device)  # [1,1,H,W]
    masks = batch["masks"].to(device)  # [1,4,H,W]
    
    # generate predictions for both models on same input
    spu_reconstructions = generate_reconstructions(spu_model, image, masks)
    hpu_reconstructions = generate_reconstructions(hpu_model, image, masks)
    
    spu_samples = generate_samples(spu_model, image, NUM_SAMPLES)
    hpu_samples = generate_samples(hpu_model, image, NUM_SAMPLES)
    
    # convert everything to numpy
    ct_scan = tensor_to_numpy_ct_image(image)
    grader_masks_np = [tensor_to_numpy_mask(masks[:, i:i+1]) for i in range(masks.shape[1])]
    
    spu_reconstructions_np = [tensor_to_numpy_mask(r) for r in spu_reconstructions]
    hpu_reconstructions_np = [tensor_to_numpy_mask(r) for r in hpu_reconstructions]
    
    spu_samples_np = [tensor_to_numpy_mask(s) for s in spu_samples]
    hpu_samples_np = [tensor_to_numpy_mask(s) for s in hpu_samples]
    
    # check what graders we have
    grader_available = [not is_empty_mask(mask) for mask in grader_masks_np]
    num_available_graders = sum(grader_available)
    total_graders = len(grader_masks_np)
    
    # compute metrics for CASE 1: all cases (overall performance)
    metrics_all_included = compute_comparative_metrics_all_cases(
        spu_reconstructions_np, hpu_reconstructions_np, grader_masks_np, GED_CLAMP_NONNEG
    )
    
    # compute metrics for CASE 2: non-empty GT only (lesion detection)
    metrics_nonempty_only = compute_comparative_metrics_nonempty_only(
        spu_reconstructions_np, hpu_reconstructions_np, grader_masks_np, GED_CLAMP_NONNEG
    )
    
    # individual IoUs for viz
    spu_ious = compute_individual_ious(spu_reconstructions_np, grader_masks_np)
    hpu_ious = compute_individual_ious(hpu_reconstructions_np, grader_masks_np)
    
    # store comprehensive metrics
    all_case_row = {
        "row_idx": row_idx,
        "total_graders": total_graders,
        "num_available_graders": num_available_graders,
        # case 1 metrics
        **{k: v for k, v in metrics_all_included.items()},
        # case 2 metrics  
        **{k: v for k, v in metrics_nonempty_only.items()}
    }
    all_case_metrics.append(all_case_row)
    
    # store for 4-grader subset
    if total_graders == 4:
        four_grader_metrics.append(all_case_row.copy())
    
    # create visualization for first N examples
    if num_visualized < NUM_EXAMPLES:
        save_path = output_dir / f"comparative_row_{row_idx:05d}.png"
        create_comparative_visualization(
            ct_scan, grader_masks_np, 
            spu_reconstructions_np, hpu_reconstructions_np,
            spu_ious, hpu_ious,
            spu_samples_np[:6], hpu_samples_np[:6],
            save_path
        )
        num_visualized += 1
    
    if EVAL_SCOPE == "visualize_only" and num_visualized >= NUM_EXAMPLES:
        break

print(f"\nevaluation complete! processed {len(all_case_metrics)} cases")

starting comparative evaluation...

processing row 0...


/tmp/ipykernel_16666/3140155420.py:12: RuntimeWarning: overflow encountered in exp
  arr = 1.0 / (1.0 + np.exp(-arr))


saved: /home/cdev/git/HPU-Net/runs/runs/iwi9140h-project/comparative_evaluation_results/comparative_row_00000.png
saved: /home/cdev/git/HPU-Net/runs/runs/iwi9140h-project/comparative_evaluation_results/comparative_row_00001.png
saved: /home/cdev/git/HPU-Net/runs/runs/iwi9140h-project/comparative_evaluation_results/comparative_row_00002.png
saved: /home/cdev/git/HPU-Net/runs/runs/iwi9140h-project/comparative_evaluation_results/comparative_row_00003.png
saved: /home/cdev/git/HPU-Net/runs/runs/iwi9140h-project/comparative_evaluation_results/comparative_row_00004.png
processing row 10...
processing row 20...
processing row 30...
processing row 40...
processing row 50...
processing row 60...
processing row 70...
processing row 80...
processing row 90...
processing row 100...
processing row 110...
processing row 120...
processing row 130...
processing row 140...
processing row 150...
processing row 160...
processing row 170...
processing row 180...
processing row 190...
processing row 200...

## Results Export
Saving detailed metrics to CSV files for further analysis

In [ ]:
def save_metrics_csv(metrics_list: List[dict], filename: str):
    """save metrics to CSV"""
    if not metrics_list:
        print(f"no data to save for {filename}")
        return
        
    csv_path = output_dir / filename
    fieldnames = list(metrics_list[0].keys())
    
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(metrics_list)
    
    print(f"saved: {csv_path}")


# save the detailed results
save_metrics_csv(all_case_metrics, "comparative_metrics_all_cases.csv")
save_metrics_csv(four_grader_metrics, "comparative_metrics_4graders_only.csv")

## Results Analysis

Here's where we compute summary statistics and see which model performs better.

We analyze both evaluation cases to understand:
1. Overall clinical performance (including empty prediction)
2. Pure lesion detection capability

In [ ]:
def print_summary_statistics(metrics_list: List[dict], label: str):
    """print summary stats for both evaluation cases"""
    if not metrics_list:
        print(f"[{label}] no data available")
        return
    
    print(f"\n{'='*80}")
    print(f"{label}")
    print(f"{'='*80}")
    print(f"total cases: {len(metrics_list)}")
    
    # case 1: all cases (overall performance)
    print(f"\n{'-'*50}")
    print(f"CASE 1: ALL CASES (Overall Performance)")
    print(f"{'-'*50}")
    
    spu_ged2_all = np.array([r["spu_GED2_all"] for r in metrics_list if not np.isnan(r["spu_GED2_all"])])
    hpu_ged2_all = np.array([r["hpu_GED2_all"] for r in metrics_list if not np.isnan(r["hpu_GED2_all"])])
    spu_iou_all = np.array([r["spu_E_IoU_SY_all"] for r in metrics_list if not np.isnan(r["spu_E_IoU_SY_all"])])
    hpu_iou_all = np.array([r["hpu_E_IoU_SY_all"] for r in metrics_list if not np.isnan(r["hpu_E_IoU_SY_all"])])
    
    print(f"valid cases: sPUNet={len(spu_ged2_all)}, HPUNet={len(hpu_ged2_all)}")
    
    if len(spu_ged2_all) > 0:
        print(f"\nsPUNet (All Cases):")
        print(f"  GED² = {np.mean(spu_ged2_all):.4f} ± {np.std(spu_ged2_all):.4f}")
        print(f"  E[IoU(S,Y)] = {np.mean(spu_iou_all):.4f} ± {np.std(spu_iou_all):.4f}")
    
    if len(hpu_ged2_all) > 0:
        print(f"\nHPUNet (All Cases):")
        print(f"  GED² = {np.mean(hpu_ged2_all):.4f} ± {np.std(hpu_ged2_all):.4f}")
        print(f"  E[IoU(S,Y)] = {np.mean(hpu_iou_all):.4f} ± {np.std(hpu_iou_all):.4f}")
    
    if len(spu_ged2_all) > 0 and len(hpu_ged2_all) > 0:
        ged2_diff_all = np.mean(hpu_ged2_all) - np.mean(spu_ged2_all)
        iou_diff_all = np.mean(hpu_iou_all) - np.mean(spu_iou_all)
        print(f"\nComparison (HPUNet - sPUNet, All Cases):")
        print(f"  ΔGED² = {ged2_diff_all:+.4f} {'(HPUNet better)' if ged2_diff_all < 0 else '(sPUNet better)'}")
        print(f"  ΔE[IoU(S,Y)] = {iou_diff_all:+.4f} {'(HPUNet better)' if iou_diff_all > 0 else '(sPUNet better)'}")
    
    # case 2: non-empty GT only (lesion detection)
    print(f"\n{'-'*50}")  
    print(f"CASE 2: NON-EMPTY GT ONLY (Lesion Detection)")
    print(f"{'-'*50}")
    
    spu_ged2_nonempty = np.array([r["spu_GED2_nonempty"] for r in metrics_list if not np.isnan(r["spu_GED2_nonempty"])])
    hpu_ged2_nonempty = np.array([r["hpu_GED2_nonempty"] for r in metrics_list if not np.isnan(r["hpu_GED2_nonempty"])])
    spu_iou_nonempty = np.array([r["spu_E_IoU_SY_nonempty"] for r in metrics_list if not np.isnan(r["spu_E_IoU_SY_nonempty"])])
    hpu_iou_nonempty = np.array([r["hpu_E_IoU_SY_nonempty"] for r in metrics_list if not np.isnan(r["hpu_E_IoU_SY_nonempty"])])
    
    print(f"valid cases: sPUNet={len(spu_ged2_nonempty)}, HPUNet={len(hpu_ged2_nonempty)}")
    
    if len(spu_ged2_nonempty) > 0:
        print(f"\nsPUNet (Non-Empty GT Only):")
        print(f"  GED² = {np.mean(spu_ged2_nonempty):.4f} ± {np.std(spu_ged2_nonempty):.4f}")
        print(f"  E[IoU(S,Y)] = {np.mean(spu_iou_nonempty):.4f} ± {np.std(spu_iou_nonempty):.4f}")
    
    if len(hpu_ged2_nonempty) > 0:
        print(f"\nHPUNet (Non-Empty GT Only):")
        print(f"  GED² = {np.mean(hpu_ged2_nonempty):.4f} ± {np.std(hpu_ged2_nonempty):.4f}")
        print(f"  E[IoU(S,Y)] = {np.mean(hpu_iou_nonempty):.4f} ± {np.std(hpu_iou_nonempty):.4f}")
    
    if len(spu_ged2_nonempty) > 0 and len(hpu_ged2_nonempty) > 0:
        ged2_diff_nonempty = np.mean(hpu_ged2_nonempty) - np.mean(spu_ged2_nonempty)
        iou_diff_nonempty = np.mean(hpu_iou_nonempty) - np.mean(spu_iou_nonempty)
        print(f"\nComparison (HPUNet - sPUNet, Non-Empty Only):")
        print(f"  ΔGED² = {ged2_diff_nonempty:+.4f} {'(HPUNet better)' if ged2_diff_nonempty < 0 else '(sPUNet better)'}")
        print(f"  ΔE[IoU(S,Y)] = {iou_diff_nonempty:+.4f} {'(HPUNet better)' if iou_diff_nonempty > 0 else '(sPUNet better)'}")


# print the summary stats
print_summary_statistics(all_case_metrics, "ALL CASES")
print_summary_statistics(four_grader_metrics, "ONLY 4-GRADER CASES")

## Export Summary Table

Creating a clean summary table that shows the key results for both evaluation approaches.

In [ ]:
print(f"\n{'='*80}")
print("COMPARATIVE EVALUATION SUMMARY")
print(f"{'='*80}")
print(f"dataset: {csv_path.name}")
print(f"evaluation scope: {EVAL_SCOPE}")
print(f"evaluation cases:")
print(f"  case 1: all cases (overall performance)")
print(f"  case 2: non-empty GT only (lesion detection)")
print(f"sPUNet checkpoint: {Path(SPUNET_CHECKPOINT).name}")
print(f"HPUNet checkpoint: {Path(HPUNET_CHECKPOINT).name}")
print(f"results saved to: {output_dir}")
print(f"visualizations created: {num_visualized}")
print(f"total cases evaluated: {len(all_case_metrics)}")
print(f"cases with 4 graders: {len(four_grader_metrics)}")

# create summary comparison table
summary_data = []
for label, metrics_list in [("All Cases", all_case_metrics), ("4-Grader Cases", four_grader_metrics)]:
    if not metrics_list:
        continue
    
    # case 1: all cases metrics
    spu_ged2_all = [r["spu_GED2_all"] for r in metrics_list if not np.isnan(r["spu_GED2_all"])]
    hpu_ged2_all = [r["hpu_GED2_all"] for r in metrics_list if not np.isnan(r["hpu_GED2_all"])]
    spu_iou_all = [r["spu_E_IoU_SY_all"] for r in metrics_list if not np.isnan(r["spu_E_IoU_SY_all"])]
    hpu_iou_all = [r["hpu_E_IoU_SY_all"] for r in metrics_list if not np.isnan(r["hpu_E_IoU_SY_all"])]
    
    # case 2: non-empty only metrics  
    spu_ged2_nonempty = [r["spu_GED2_nonempty"] for r in metrics_list if not np.isnan(r["spu_GED2_nonempty"])]
    hpu_ged2_nonempty = [r["hpu_GED2_nonempty"] for r in metrics_list if not np.isnan(r["hpu_GED2_nonempty"])]
    spu_iou_nonempty = [r["spu_E_IoU_SY_nonempty"] for r in metrics_list if not np.isnan(r["spu_E_IoU_SY_nonempty"])]
    hpu_iou_nonempty = [r["hpu_E_IoU_SY_nonempty"] for r in metrics_list if not np.isnan(r["hpu_E_IoU_SY_nonempty"])]
    
    if (spu_ged2_all and hpu_ged2_all and spu_iou_all and hpu_iou_all and 
        spu_ged2_nonempty and hpu_ged2_nonempty and spu_iou_nonempty and hpu_iou_nonempty):
        
        summary_data.extend([
            {
                "Dataset": f"{label}_AllCases",
                "Evaluation_Type": "All Cases Included", 
                "sPUNet_GED2_mean": np.mean(spu_ged2_all),
                "sPUNet_GED2_std": np.std(spu_ged2_all),
                "HPUNet_GED2_mean": np.mean(hpu_ged2_all), 
                "HPUNet_GED2_std": np.std(hpu_ged2_all),
                "sPUNet_IoU_mean": np.mean(spu_iou_all),
                "sPUNet_IoU_std": np.std(spu_iou_all),
                "HPUNet_IoU_mean": np.mean(hpu_iou_all),
                "HPUNet_IoU_std": np.std(hpu_iou_all),
            },
            {
                "Dataset": f"{label}_NonEmptyOnly", 
                "Evaluation_Type": "Non-Empty GT Only",
                "sPUNet_GED2_mean": np.mean(spu_ged2_nonempty),
                "sPUNet_GED2_std": np.std(spu_ged2_nonempty),
                "HPUNet_GED2_mean": np.mean(hpu_ged2_nonempty), 
                "HPUNet_GED2_std": np.std(hpu_ged2_nonempty),
                "sPUNet_IoU_mean": np.mean(spu_iou_nonempty),
                "sPUNet_IoU_std": np.std(spu_iou_nonempty),
                "HPUNet_IoU_mean": np.mean(hpu_iou_nonempty),
                "HPUNet_IoU_std": np.std(hpu_iou_nonempty),
            }
        ])

if summary_data:
    save_metrics_csv(summary_data, "summary_comparison.csv")
    print(f"\nsummary comparison saved to: {output_dir}/summary_comparison.csv")

## Final Summary

Wrapping up with a final overview of what we accomplished.

In [ ]:
print("\n🎉 comparative evaluation complete!")
print("\nthis notebook evaluated both models using:")
print("  📊 case 1: all cases (overall performance including empty prediction)")
print("  🎯 case 2: non-empty GT only (lesion detection when lesions present)")
print(f"\ncheck {output_dir}/ for:")
print("  • detailed per-case metrics (CSV files)")
print("  • side-by-side visualizations (PNG files)")
print("  • summary comparison table")

# quick final comparison if we have data
if all_case_metrics:
    # get overall winner for each case
    print("\n" + "="*50)
    print("QUICK SUMMARY")
    print("="*50)
    
    # case 1 winner
    spu_ged2_all = [r["spu_GED2_all"] for r in all_case_metrics if not np.isnan(r["spu_GED2_all"])]
    hpu_ged2_all = [r["hpu_GED2_all"] for r in all_case_metrics if not np.isnan(r["hpu_GED2_all"])]
    
    if spu_ged2_all and hpu_ged2_all:
        case1_winner = "HPUNet" if np.mean(hpu_ged2_all) < np.mean(spu_ged2_all) else "sPUNet"
        print(f"overall performance (case 1): {case1_winner} wins")
    
    # case 2 winner
    spu_ged2_nonempty = [r["spu_GED2_nonempty"] for r in all_case_metrics if not np.isnan(r["spu_GED2_nonempty"])]
    hpu_ged2_nonempty = [r["hpu_GED2_nonempty"] for r in all_case_metrics if not np.isnan(r["hpu_GED2_nonempty"])]
    
    if spu_ged2_nonempty and hpu_ged2_nonempty:
        case2_winner = "HPUNet" if np.mean(hpu_ged2_nonempty) < np.mean(spu_ged2_nonempty) else "sPUNet"
        print(f"lesion detection (case 2): {case2_winner} wins")

print("\ndone! 🚀")